In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# **1. Reading and exploring the data**

In [3]:
with open('./data/input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [4]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [5]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [6]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [7]:
# create a mapping from characters to integers
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integer
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [8]:
# let's now encode the entire text dataset and store it in torch.tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [9]:
# let's now split the data into train and val sets
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [10]:
block_size = 8
print(decode(train_data[:block_size].tolist()))
print(decode(train_data[:block_size + 1].tolist()))

First Ci
First Cit


In [11]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
  context = x[:t + 1]
  target = y[t]
  print(f'when input is {context} the target: {target}')

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [12]:
ix = torch.randint(5, (1, 10))
print(ix, ix.shape)

tensor([[0, 2, 2, 0, 0, 3, 0, 0, 4, 0]]) torch.Size([1, 10])


In [13]:
data[200: 208], data[201: 209]

(tensor([ 1, 49, 52, 53, 61,  1, 15, 39]),
 tensor([49, 52, 53, 61,  1, 15, 39, 47]))

In [14]:
batch_size = 4 # how many independent sequences will we process in parallel ?
block_size = 8 # what is the maximum context length for predictions ?

def get_batch(split):
  # generate small batch of data of inputs x and target y
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i: i + block_size] for i in ix]) # stack by row to make it a (batch_size, block_size) vector
  y = torch.stack([data[i + 1: i + block_size + 1] for i in ix]) # stack by row to make it a (batch_size, block_size) vector
  return x, y

xb, yb = get_batch('train')
print('inputs(our input to transformer):',xb.shape, '-->', xb)
print('targets:', yb.shape, '-->', yb)
print('---' * 7)

for b in range(batch_size): # block dimension 
  for t in range(block_size): # time dimension
    context = xb[b, :t + 1]
    target = yb[b, t]
    print(f"when input is {context.tolist()} the target: {target}")


inputs(our input to transformer): torch.Size([4, 8]) --> tensor([[39, 58, 47, 53, 52, 12,  1, 37],
        [53, 56, 43,  1, 21,  1, 41, 39],
        [50, 39, 52, 63,  1, 47, 58, 57],
        [56, 53, 63,  1, 42, 47, 42,  1]])
targets: torch.Size([4, 8]) --> tensor([[58, 47, 53, 52, 12,  1, 37, 53],
        [56, 43,  1, 21,  1, 41, 39, 51],
        [39, 52, 63,  1, 47, 58, 57, 43],
        [53, 63,  1, 42, 47, 42,  1, 57]])
---------------------
when input is [39] the target: 58
when input is [39, 58] the target: 47
when input is [39, 58, 47] the target: 53
when input is [39, 58, 47, 53] the target: 52
when input is [39, 58, 47, 53, 52] the target: 12
when input is [39, 58, 47, 53, 52, 12] the target: 1
when input is [39, 58, 47, 53, 52, 12, 1] the target: 37
when input is [39, 58, 47, 53, 52, 12, 1, 37] the target: 53
when input is [53] the target: 56
when input is [53, 56] the target: 43
when input is [53, 56, 43] the target: 1
when input is [53, 56, 43, 1] the target: 21
when input i

***

# **2. Simplest baseline: bigram language model, loss, generation**

In [15]:
class BigramLanguageModel(nn.Module):

  def __init__(self, vocab_size):

    super().__init__()
    # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets = None): # idx --> xb, targets --> yb

    # idx and targets are both (B, T) tensor of integers
    logits = self.token_embedding_table(idx) # (B, T, C) # B - Block - batch_size, T - Time - block_size, C - Channel - vocab_size

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B * T, C) # re-shaping to match pytorch cross entropy definition
      targets = targets.view(B * T) # re-shaping to match pytorch cross entropy definition
      loss = F.cross_entropy(logits, targets)

    return logits, loss
  
  def generate(self, idx, max_new_tokens):

    # idx is (B, T) array of indices in current context
    for _ in range(max_new_tokens):
      # print(idx)
      # get the predictions
      logits, loss = self(idx) # shape of logits (1, 1, 65), (1, 2, 65)....(1, max_new_tokens, 65)
      
      # focus only on the last time step
      logits = logits[:, -1, :] # becomes (B, C)

      # apply softmax to get probabilities
      probs = F.softmax(logits, dim=1) # (B, C)

      # sample from the distribution
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

      # append the sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T + 1)
    
    return idx

  
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(f'{loss=}')

# m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100) shape -> [1, 101]
# m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0] shape -> [101]
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
loss=tensor(4.7899, grad_fn=<NllLossBackward0>)


aBVFMWPVwEF.MIoSuaQ'g?Md:!F;;vjMHDxRt&kCI
hObEdBcZF;sU:HnqlUEpSLXQ'3HuXN?i;pgREfgp?PkXnlg3TQ:HlBEJC


In [19]:
logits, logits.shape

(tensor([[-0.6473, -0.1628, -1.3359,  ..., -0.2257,  1.7677,  0.2006],
         [-0.3812, -0.8515,  1.1918,  ...,  0.0947,  0.9018,  0.7659],
         [ 1.1190, -0.3601,  1.7839,  ...,  1.2128,  1.3650, -1.3300],
         ...,
         [ 1.1190, -0.3601,  1.7839,  ...,  1.2128,  1.3650, -1.3300],
         [-1.3560, -1.5675, -0.5239,  ...,  0.9869, -0.7510,  1.6773],
         [ 0.9154,  2.4900, -1.2237,  ..., -0.7472,  0.0936, -0.8834]],
        grad_fn=<ViewBackward0>),
 torch.Size([32, 65]))

In [22]:
logits[..., :-1, :], logits[..., :-1, :].shape, logits[..., :-1, :].size(-1)

(tensor([[-0.6473, -0.1628, -1.3359,  ..., -0.2257,  1.7677,  0.2006],
         [-0.3812, -0.8515,  1.1918,  ...,  0.0947,  0.9018,  0.7659],
         [ 1.1190, -0.3601,  1.7839,  ...,  1.2128,  1.3650, -1.3300],
         ...,
         [-1.3560, -1.5675, -0.5239,  ...,  0.9869, -0.7510,  1.6773],
         [ 1.1190, -0.3601,  1.7839,  ...,  1.2128,  1.3650, -1.3300],
         [-1.3560, -1.5675, -0.5239,  ...,  0.9869, -0.7510,  1.6773]],
        grad_fn=<SliceBackward0>),
 torch.Size([31, 65]),
 65)

In [21]:
logits[..., 1:], logits[..., 1:].shape

(tensor([[-0.1628, -1.3359, -1.2415,  ..., -0.2257,  1.7677,  0.2006],
         [-0.8515,  1.1918, -0.8108,  ...,  0.0947,  0.9018,  0.7659],
         [-0.3601,  1.7839,  0.2669,  ...,  1.2128,  1.3650, -1.3300],
         ...,
         [-0.3601,  1.7839,  0.2669,  ...,  1.2128,  1.3650, -1.3300],
         [-1.5675, -0.5239,  0.6937,  ...,  0.9869, -0.7510,  1.6773],
         [ 2.4900, -1.2237,  1.0107,  ..., -0.7472,  0.0936, -0.8834]],
        grad_fn=<SliceBackward0>),
 torch.Size([32, 64]))

In [24]:
shift_logits = (logits[..., :-1, :]).contiguous()
shift_logits, shift_logits.shape

(tensor([[-0.6473, -0.1628, -1.3359,  ..., -0.2257,  1.7677,  0.2006],
         [-0.3812, -0.8515,  1.1918,  ...,  0.0947,  0.9018,  0.7659],
         [ 1.1190, -0.3601,  1.7839,  ...,  1.2128,  1.3650, -1.3300],
         ...,
         [-1.3560, -1.5675, -0.5239,  ...,  0.9869, -0.7510,  1.6773],
         [ 1.1190, -0.3601,  1.7839,  ...,  1.2128,  1.3650, -1.3300],
         [-1.3560, -1.5675, -0.5239,  ...,  0.9869, -0.7510,  1.6773]],
        grad_fn=<SliceBackward0>),
 torch.Size([31, 65]))

In [25]:
shift_logits.view(-1, shift_logits.size(-1))

tensor([[-0.6473, -0.1628, -1.3359,  ..., -0.2257,  1.7677,  0.2006],
        [-0.3812, -0.8515,  1.1918,  ...,  0.0947,  0.9018,  0.7659],
        [ 1.1190, -0.3601,  1.7839,  ...,  1.2128,  1.3650, -1.3300],
        ...,
        [-1.3560, -1.5675, -0.5239,  ...,  0.9869, -0.7510,  1.6773],
        [ 1.1190, -0.3601,  1.7839,  ...,  1.2128,  1.3650, -1.3300],
        [-1.3560, -1.5675, -0.5239,  ...,  0.9869, -0.7510,  1.6773]],
       grad_fn=<ViewBackward0>)

***
**Confused with shapes!! Aaaaah!!!!!! Let's figure out**

In [ ]:
a = torch.randint(5, (5, 5))
print(a)
print(a.shape)

tensor([[0, 4, 0, 2, 4],
        [2, 1, 1, 0, 3],
        [0, 3, 4, 0, 0],
        [0, 1, 2, 3, 3],
        [0, 2, 1, 3, 0]])
torch.Size([5, 5])
tensor([[4, 0, 2, 4],
        [1, 1, 0, 3],
        [3, 4, 0, 0],
        [1, 2, 3, 3],
        [2, 1, 3, 0]])


In [17]:
idx = torch.tensor([[0, 1, 2]])
print(a[idx].shape)
print(a[idx])


torch.Size([1, 3, 5])
tensor([[[0, 4, 0, 2, 4],
         [2, 1, 1, 0, 3],
         [0, 3, 4, 0, 0]]])


In [18]:
e = nn.Embedding(65, 65)
idx = torch.tensor([[0]])
print(e(idx).shape)



idx = torch.tensor([[0, 1, 2]])
print(e(idx).shape)

idx = torch.tensor([[0, 1, 2, 3, 4]])
print(e(idx).shape)

print(e(idx))
print(e(idx)[:, -1, :])
print(e(idx)[:, -1, :].shape)
print(e(idx).view(1 * 5, 65).shape)

torch.Size([1, 1, 65])
torch.Size([1, 3, 65])
torch.Size([1, 5, 65])
tensor([[[-0.4039, -0.4708,  0.2819, -2.2789, -0.1748, -0.4633, -0.8990,
          -0.7207, -0.3893,  0.0058,  0.3379,  2.4657,  0.9794, -0.3083,
           0.4546,  0.4213, -0.4884, -2.2580, -0.6790, -0.5462,  0.8102,
          -0.5445,  0.5627,  0.1699,  0.0742,  0.6964, -0.5413,  2.8676,
          -0.2098, -0.9856,  0.7624,  1.8034,  0.2011,  0.8107, -0.3692,
           0.1770,  0.8852,  0.6048, -0.0746,  0.8823,  0.8340, -0.0658,
           0.7882, -0.5568,  0.5584, -0.8097,  0.6547, -0.6994, -1.9133,
           0.3502,  0.2631, -0.6641,  1.6919, -1.3639, -1.0970, -0.0552,
          -1.1828,  0.9248,  0.4386,  0.5336, -0.3136, -0.4317, -0.2513,
           0.4203, -0.7798],
         [ 1.5507,  0.0947, -0.7500, -1.8097,  0.1889, -0.9150, -0.7925,
           0.4724,  0.4832, -0.2741,  0.2768, -0.1760,  0.0093, -1.9002,
           1.3534,  0.8204, -0.2594,  0.6513,  0.5492,  0.7804, -1.0058,
           0.1144, -0.1422

In [19]:
a = torch.randint(5, (5, 7))
print(f'{a=}')
b = torch.randint(5, (5, 1))
print(f'{b=}')
torch.cat((a, b), dim=1)

a=tensor([[4, 0, 1, 2, 1, 1, 1],
        [0, 3, 4, 4, 3, 1, 4],
        [3, 4, 0, 4, 1, 0, 3],
        [4, 3, 0, 1, 1, 4, 0],
        [2, 4, 3, 0, 2, 0, 1]])
b=tensor([[0],
        [1],
        [4],
        [3],
        [0]])


tensor([[4, 0, 1, 2, 1, 1, 1, 0],
        [0, 3, 4, 4, 3, 1, 4, 1],
        [3, 4, 0, 4, 1, 0, 3, 4],
        [4, 3, 0, 1, 1, 4, 0, 3],
        [2, 4, 3, 0, 2, 0, 1, 0]])

**No more confusion**
***

# **3. Training the bigram model**

In [20]:
# create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [22]:
batch_size = 32
for steps in range(10000):

  # sample batch of data
  xb, yb = get_batch('train')

  # evaluate the loss
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.4488911628723145


In [23]:
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


't weristheandsk.
Y:
Fo VIUS:
IO, he f ju'd-ws y m.
SARDIAnd fuco m, cor that y cere te d ltaven t pl;
THESSThen
ARUSThango wn BRUCI amyou if t, d; hy wesserd,
PUThef bssend,
Cowhise
QUKI withow fr d by,

VI ch he l--d nomy t s ho, lorerenttl wavishes w,
And dy imay ipl h t thoury, sts ne mmede fos 


# **4. Building the "self-attention"**

## 4.1 Version 1: averaging past context with for loops, the weakest form of aggregation

In [25]:
# consider this toy example
B, T, C = 4, 8, 2 # batch, time, channel
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [29]:
print(x)
print(x[2, :5])

tensor([[[-0.5262,  0.6526],
         [ 1.1759, -0.1322],
         [ 1.1763, -1.9545],
         [-0.9405, -0.1877],
         [ 0.8490,  1.1114],
         [-0.1556,  0.9508],
         [-2.1512, -0.8620],
         [ 1.7051, -1.4240]],

        [[-1.1872, -1.0207],
         [-1.4724, -0.0915],
         [ 0.9664, -0.6772],
         [-0.7393,  0.9323],
         [ 1.9494,  0.2143],
         [-0.6531,  0.0098],
         [ 1.8095, -0.0395],
         [-0.5375, -1.4407]],

        [[-1.4393, -1.3821],
         [ 0.4619, -0.6969],
         [-0.1959,  0.4177],
         [-0.8843, -1.0004],
         [ 1.0674, -1.2204],
         [-1.5746,  0.5919],
         [-1.8647,  0.6649],
         [ 0.4639, -0.3095]],

        [[-0.2135, -1.1805],
         [-0.6327, -0.0114],
         [-0.5840,  0.3981],
         [ 0.2183,  1.2908],
         [-0.4561,  0.1044],
         [-1.7859, -0.2099],
         [-2.4274,  0.5819],
         [ 0.7469,  0.6079]]])
tensor([[-1.4393, -1.3821],
        [ 0.4619, -0.6969],
        

So currently we have 8 tokens in a batch, these 8 tokens are not talking to each other and we would like them to talk to each other. We want to couple them in a very specific way.

For example the 5th token should not talk to 6th 7th and 8th token because they are future tokens in the sequence, they should only talk to 4th 3rd 2nd and 1st.
Now what we like to do is:

- For every single batch elements, for every tth token, we would like to calculate the average of all the vectors in all the previous tokens and also at this token.

In [ ]:
# we want x[b, t] = mean_{i <= t} x[b, i]
xbow = torch.zeros((B, T, C)) # bow -> bag of words
for b in range(B):
  for t in range(T):
    xprev = x[b, :t+1] # (t, C)
    xbow[b, t] = torch.mean(xprev, 0)

In [31]:
x[0], xbow[0]

(tensor([[-0.5262,  0.6526],
         [ 1.1759, -0.1322],
         [ 1.1763, -1.9545],
         [-0.9405, -0.1877],
         [ 0.8490,  1.1114],
         [-0.1556,  0.9508],
         [-2.1512, -0.8620],
         [ 1.7051, -1.4240]]),
 tensor([[-0.5262,  0.6526],
         [ 0.3248,  0.2602],
         [ 0.6086, -0.4780],
         [ 0.2214, -0.4054],
         [ 0.3469, -0.1021],
         [ 0.2631,  0.0734],
         [-0.0818, -0.0602],
         [ 0.1416, -0.2307]]))

Now of course for loop is very inefficient, we can do this using matrix multiplication.

## 4.2 Version 2: The trick in self-attention: matrix multiply as weighted aggregation

In [39]:
torch.tril(torch.ones(3, 3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [38]:
torch.manual_seed(42)
a = torch.ones(3, 3)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b 
print('a=')
print(a)
print('b=')
print(b)
print('c=')
print(c)

a=
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


In [40]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b 
print('a=')
print(a)
print('b=')
print(b)
print('c=')
print(c)

a=
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


So depending on the triangular matrix we notice that we are summing the first elements, then first two elements, then first 3 elements, this is the magic of triangular matrix.

**Output vector C**

- [ 2                     7  ]
- [(2 + 6)       (    7 + 4) ]
- [(2 + 6 + 6)   (7 + 4 + 5) ]

In [41]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
# wooooh!!!! trick!!!
a /= torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b 
print('a=')
print(a)
print('b=')
print(b)
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


**Output vector C**

- [ 2                     7  ]
- [(2 + 6) / 2       (    7 + 4) /2 ]
- [(2 + 6 + 6) /3   (7 + 4 + 5) /3 ]

**Let's return back to our original implementation**

In [42]:
wei = torch.tril(torch.ones(T, T))
wei /= torch.sum(wei, 1, keepdim=True)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [44]:
xbow2 = wei @ x # (T, T) X (B, T, C) -> (B, T, T) X (B, T, C) broadcasting -> (B, T, C)
xbow2.shape

torch.Size([4, 8, 2])

In [45]:
torch.allclose(xbow, xbow2)

True

## 4.3 Version 3: Adding softmax

In [60]:
wei = torch.zeros(T, T)
wei

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [61]:
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf')) # wherever tril is equal to 0, fill it with -inf
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [62]:
wei = F.softmax(wei, dim=1) # softmax is essentially exponentiation of all the numbers then divide by sum of the exponentiation
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [63]:
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

**bringing it all together**

In [64]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf')) # wherever tril is equal to 0, fill it with -inf
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


True

We are going to use this method in self-attention.

- wei = torch.zeros(T, T) --> Weights(wei) begin with 0 and we can think of these as interaction strength or like an affinity, it tell us how much of the token from the past we want to aggregate or average up.

- wei = wei.masked_fill(tril == 0, float('-inf')) --> token from the past cannot communicate with token from the future by setting them to -inf, future cannot communicate with the past

Thus we can do weighted aggregation of the past elements by using matrix multiplication of a lower triangular matrix and then elements of the lower triangular matrix tells us how much each element fuses into this position

## 4.4 Version 4: self-attention

In [71]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32 # batch, time, channels
x = torch.randn(B, T, C)

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

out.shape

torch.Size([4, 8, 32])

wei = torch.zeros((T, T)) -> we do not want this to be actually uniform, as tokens will find other tokens more or less interesting and we want that to be data dependent.
So now we want to gather information from the past but we want to do it in a data dependent way and this is the problem that self attention solves.

Self attention for every single node or every single token at each position will emit two vectors - query and key

In the attention mechanism of machine learning models (especially Transformers), query (Q), key (K), and value (V) vectors work like a database search to determine which parts of the input are most relevant to each other

In a neural network, each word (token) in the input sequence is converted into an embedding vector. These embeddings are then multiplied by three different, learned weight matrices (WQ,WK,WV) to create unique Query, Key, and Value vectors for each word. 

- Query: A vector representation of the current word that is used to "ask" for relevant information from all other words.
- Key: A vector representation for each word that describes what information it "offers". The similarity between the Query and all Keys is calculated using a dot product to produce attention scores.
- Value: A vector representation of the actual content or meaning of a word. These are weighted by the attention scores (via a softmax function for normalization) and summed to create a new, context-aware representation for the original query word. 

This process allows the model to dynamically focus on relevant parts of the input sequence, capturing context and dependencies between words, which is crucial for tasks like machine translation and text generation. 

In [74]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32 # batch, time, channels
x = torch.randn(B, T, C)

# let's see a single Head perform in self attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, head_size) -> (B, T, 16)
q = query(x) # (B, T, head_size) -> (B, T, 16)
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) -> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

In [76]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [77]:
out[0]

tensor([[ 0.1808, -0.0700, -0.3596, -0.9152,  0.6258,  0.0255,  0.9545,  0.0643,
          0.3612,  1.1679, -1.3499, -0.5102,  0.2360, -0.2398, -0.9211,  1.5433,
          1.3488, -0.1396,  0.2858,  0.9651, -2.0371,  0.4931,  1.4870,  0.5910,
          0.1260, -1.5627, -1.1601, -0.3348,  0.4478, -0.8016,  1.5236,  2.5086],
        [-0.5303, -0.2227,  0.7946, -0.0416,  0.2320,  0.9596, -0.8221, -0.2413,
         -0.3708, -0.5947,  0.2482, -1.3398, -0.9788,  0.4441, -0.6483, -0.3416,
          1.5988, -0.6986,  1.1837, -0.0806, -1.5937,  1.8511,  2.5621, -1.3786,
          1.2430, -1.5185,  0.5093, -0.2309,  0.7268,  1.1658,  1.5962,  0.0550],
        [-0.5943,  0.3186,  0.0590, -0.2116, -0.1547,  0.4838, -0.1518, -0.7044,
          1.2507, -0.2447, -0.0523,  0.0842, -1.0431,  0.6117, -0.2001,  0.3510,
          2.1127, -0.9281,  0.9154,  0.5045, -1.6725, -0.3468,  1.1978, -0.2869,
          0.4814, -0.7891,  0.1911, -0.5849, -0.0408, -0.1889, -0.0443,  0.2009],
        [-0.3202, -0.1119

In [79]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32 # batch, time, channels
x = torch.randn(B, T, C)

# let's see a single Head perform in self attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, head_size) -> (B, T, 16)
q = query(x) # (B, T, head_size) -> (B, T, 16)
v = value(x) # (B, T, head_size) -> (B, T, 16)

wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) -> (B, T, T)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

out = wei @ v # (B, T, 16)
out.shape

torch.Size([4, 8, 16])

<img src="./imgs/Screenshot 2025-12-27 at 07.50.43.png" width="50%"/>

**Notes: -**

- Attention is a communication mechanism, it can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data dependent weights.
  - In our case we have 8 nodes because our block size is 8. The first node is pointed to itself, whereas the second node is pointed to first node and itself and eventually all the way to the 8th token/node pointed to all the nodes and itself.

- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.

- Each example across batch dimension is of course processed completely independently and never "talk" to each other

- In an "encoder" attention block just delete the single line that does masking with tril, allowing tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in an auto-regressive settings, like language modelling.

- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)

- "Scaled" attention additional divides wei by 1/sqrt(head_size) dk from the above image. This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [86]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
wei = q @ k.transpose(-2, -1)

In [87]:
k.var(), q.var(), wei.var()

(tensor(1.0104), tensor(1.0204), tensor(17.6841))

We notice that we have a variance comparable to head_size

In [88]:
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [89]:
k.var(), q.var(), wei.var()

(tensor(1.0104), tensor(1.0204), tensor(1.1053))

Now when we divide wei by sqrt of head_size we see that it is of unit variance. Why this is important?

- As wei feeds to softmax if wei takes on very positive and very negative numbers inside it softmax will actually converge towards one hot vectors.

In [92]:
# Code to the above explanation
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

We can see that the tensor feeding to softmax has values close to 0 and the output of softmax is diffused

In [94]:
# as soon as we sharpen it let's say make it larger
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]) * 8, dim=-1)

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

Now when we multiply these numbers by 8, we notice that softmax sharpens to the number which is highest -> one hot encoding

# **5. Layernorm (and its relationship to batchnorm)**

In [117]:
class BatchNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True

    # parameters - trained with back prop
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

    # buffers - trained with a running momentum update
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim) # std dev

  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True, unbiased=True) # batch variance

    else:
      xmean = self.running_mean
      xvar = self.running_var

    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta

    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
      
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

In [118]:
torch.manual_seed(1337)
module = BatchNorm1d(100)

x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [119]:
x

tensor([[ 0.0468,  0.5465,  0.0375,  ..., -0.9553, -0.3555,  1.2508],
        [-0.1209, -0.1865,  0.1040,  ..., -0.3002, -0.6925,  0.8246],
        [-0.1358, -1.1891,  1.6255,  ..., -0.5492,  0.5944,  0.7645],
        ...,
        [-2.0062, -1.2021, -0.3530,  ..., -0.6056, -0.0319, -0.6835],
        [-0.4110,  0.6467,  0.1247,  ..., -0.5573, -1.6659, -0.6352],
        [ 0.0635,  0.3166, -0.1367,  ..., -1.2361, -1.2328, -1.1261]])

In [120]:
x[:, 0].mean(), x[:, 0].std() # mean, std of one feature across all batch inputs

(tensor(1.4901e-08), tensor(1.0000))

In [121]:
x[0, :].mean(), x[0, :].std() # mean, std of a single input from the batch, of it's features

(tensor(0.0411), tensor(1.0431))

so we just normalize across the columns, we do not normalize across the rows

In [122]:
class BatchNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True

    # parameters - trained with back prop
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

    # buffers - trained with a running momentum update
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim) # std dev

  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(1, keepdim=True) # batch mean
      xvar = x.var(1, keepdim=True, unbiased=True) # batch variance

    else:
      xmean = self.running_mean
      xvar = self.running_var

    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta

    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
      
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

In [123]:
torch.manual_seed(1337)
module = BatchNorm1d(100)

x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [124]:
x[:, 0].mean(), x[:, 0].std() # mean, std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [125]:
x[0, :].mean(), x[0, :].std() # mean, std of a single input from the batch, of it's features

(tensor(2.3842e-09), tensor(1.0000))

Now rows are normalized, columns are not normalized, so for every indv examples all 100 vectors are normalized, and because our computation does not span across examples(column wise normalization is not present here), we can remove these buffers.

### Final Implementation of Layer Norm

In [126]:
class LayerNorm:
  # https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.normalization.LayerNorm.html
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps

    # parameters - trained with back prop
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True, unbiased=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]